In [1]:
## IMPORTS

import numpy as np
import pandas as pd
import geopandas as gpd
import xarray as xr
import rioxarray
import matplotlib.pyplot as plt
import os
import dask
import dask.array

import datetime

from collections import Counter

import pystac_client
from pystac.extensions.projection import ProjectionExtension as proj

import planetary_computer
import rasterio
import rasterio.features
from rasterio.features import rasterize

import stackstac
import pyproj

import dask.diagnostics

from shapely.geometry import box
from shapely.ops import transform

from scipy.ndimage import binary_propagation
from scipy.ndimage import label



In [2]:
catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=planetary_computer.sign_inplace,
)


In [3]:
## FUNCTION TO DEFINE BOUNDING BOX AROUND A GIVEN CENTROID
def bounds_latlon_around(center_lon, center_lat, side_m=10000):
    """
    center_lon, center_lat : centroid in decimal degrees (EPSG:4326)
    side_m                 : length of box side in meters (default 10 km)
    returns                 : (minx, miny, maxx, maxy) in lon/lat
    """
    # 1) set up transformers
    to_ps = pyproj.Transformer.from_crs(4326, 3413, always_xy=True).transform
    to_ll = pyproj.Transformer.from_crs(3413, 4326, always_xy=True).transform

    # 2) project centroid into EPSG:3413 (units = m)
    x0, y0 = to_ps(center_lon, center_lat)

    # 3) build a square of side `side_m` centered on (x0,y0)
    half = side_m / 2.0
    sq_m = box(x0 - half, y0 - half, x0 + half, y0 + half)

    # 4) reproject that square back to lon/lat and grab its bounds
    sq_ll = transform(to_ll, sq_m)
    return sq_ll.bounds  # (minx, miny, maxx, maxy)




In [4]:
## APPLY FUNCTION TO GET BOUNDING BOX OF CERTAIN SIZE AROUND CENTROID OF INTEREST
centroid = (-49.495, 68.725)       # lon, lat of North Lake, for example
# centroid = (-49.26457,68.63694)
bounds_latlon = bounds_latlon_around(*centroid, side_m=20000) # box side length
print(bounds_latlon)



(-49.7604142037196, 68.62884338160838, -49.23166982982294, 68.82086243951774)


In [5]:
## GET COLLECTION FOR THE AREA OF INTEREST (BOX AROUND CENTROID) DEFINED ABOVE

time_range = '2019-05-01/2019-09-30'
bbox = bounds_latlon

band_names = ["B04",  # red (665 nm)
              "B03",  # green (560 nm)
              "B02",  # blue (490 nm)
              "B08",  # NIR (842 nm)
              "B11"]  # SWIR1 (1610 nm)

search = catalog.search(
    collections=["sentinel-2-l2a"],
    bbox=bbox,
    datetime=time_range,
    query={"eo:cloud_cover": {"lt": 20}}
)

items = search.item_collection()

print(f'There are {len(items)} images that fit your search criteria in the database.')
print(type(items))



There are 77 images that fit your search criteria in the database.
<class 'pystac.item_collection.ItemCollection'>


In [6]:
## CHECK FOR DUPLICATE DATES
# 1) extract calendar dates (YYYY-MM-DD) from your STAC items
date_list = [item.datetime.date().isoformat() for item in items]

# 2) count how often each date occurs
counts = Counter(date_list)

# 3) find any repeats
duplicates = [d for d, c in counts.items() if c > 1]

# 4) raise or print
if duplicates:
    raise ValueError(f"Duplicate dates found in items: {duplicates}")
# else, all good


In [7]:
## FUNCTION TO GET LISTS OF DATES WITH CLEAR TILES, CLOUDY TILES, AND BLANK TILES AS WELL AS A VECTOR OF 0,1,2 REPRESENTING CLEAR,CLOUDY,BLANK TILES IN THE SEQUENCE
# dates_cleartiles = [item.datetime.date().isoformat() for item in items]

def dates_clearcloudyblank(cloud_limit, collection_items):

    # parse calendar range
    start_str, end_str = time_range.split("/")
    start_date = datetime.datetime.fromisoformat(start_str).date()
    end_date   = datetime.datetime.fromisoformat(end_str).date()
    n_days   = (end_date - start_date).days + 1
    
    # build ordered list of all dates in calendar range
    all_dates = [
        (start_date + datetime.timedelta(days=i)).isoformat()
        for i in range(n_days)
    ]

    # split colleted dates into clear/cloudy
    dates_clear  = []
    dates_cloudy = []
    for item in items:
        d = item.datetime.date().isoformat()
        if item.properties["eo:cloud_cover"] < cloud_limit:
            dates_clear.append(d)
        else:
            dates_cloudy.append(d)

    # compute missing dates (will later become blank tiles)
    seen = set(dates_clear) | set(dates_cloudy)
    dates_blank = [d for d in all_dates if d not in seen]

    # build ccb_vec with entries of 0 (clear day), 1 (cloudy day), 2 (missing/blank day)
    ccb_list = ccb_vec = [
        0 if d in dates_clear
        else 1 if d in dates_cloudy
        else 2
        for d in all_dates
    ]
    ccb = np.array(ccb_list)

    # raise warning if there are repeated dates


    return dates_clear, dates_cloudy, dates_blank, ccb


In [8]:
## APPLY FUNCTION TO GET LISTS OF DATES WITH CLEAR TILES, CLOUDY TILES, AND BLANK TILES

cloud_limit = 50 #[percent]
dates_clear, dates_cloudy, dates_blank, ccb = dates_clearcloudyblank(cloud_limit, items)

print(f"number of clear dates: {len(dates_clear)}")
print(f"number of cloudy dates: {len(dates_cloudy)}")
print(f"number of missing dates: {len(dates_blank)}")
print(f"total dates: {len(dates_cloudy)+len(dates_clear)+len(dates_blank)}")
print(f"len(ccb) = {len(ccb)}")


number of clear dates: 77
number of cloudy dates: 0
number of missing dates: 76
total dates: 153
len(ccb) = 153


In [9]:
## STACK COLLECTION ITEMS

# create the stack
stack = stackstac.stack(items, 
                        epsg=proj.ext(items[0]).epsg, 
                        assets=['B04','B03','B02', 'B08', 'B11'],
                        bounds_latlon=bounds_latlon, 
                        resolution=10,
                        chunksize=(1, 1, 4096, 4096),
                        )

mask_edge = (stack == 0).all(dim='band')
stack = stack.where(~mask_edge)
# stack



In [10]:
# FIRST NORMALIZE EACH IMAGE IN STACK BEFORE TAKING WEEKLY RESAMPLE
def combo_scaler(x, range_max=1):
    median_x = np.nanmedian(x)
    iqr_x = np.nanpercentile(x,75) - np.nanpercentile(x,25)
    robust_x = ((x-median_x)/iqr_x)
    
    return ((robust_x - np.nanmin(robust_x)) / (np.nanmax(robust_x) - np.nanmin(robust_x))) * range_max


stack_rechunk = stack.chunk({'y': -1, 'x': -1})

stack_scaled = xr.apply_ufunc(
    combo_scaler,            # combo_scaler function
    stack_rechunk,                   # DataArray stack
    input_core_dims=[['y','x']],
    output_core_dims=[['y','x']],
    vectorize=True,          # loop over time & band dims
    dask='parallelized',
    output_dtypes=[float],
    kwargs={'range_max':1},
    keep_attrs=True
)


In [11]:
## DO WEEKLY AVERAGING

# NOW TAKE WEEKLY RESAMPLE
weekly = stack_scaled.resample(time="W").mean("time", keep_attrs=True)
# weekly

# weekly_scaled = stack_scaled.resample(time="W").mean("time", keep_attrs=True)
# weekly_scaled


In [12]:
## SPECIFY TILE OF SPECIFIED PIXEL DIMENSION (e.g., 512x512) TO CROP OUT OF THE LARGER rgb STACK

tile_size = 2048 #[pix]
pix_res = 10 #[m/pix]
buffer = tile_size * pix_res/2 #[m]
x_utm, y_utm = pyproj.Proj(stack.crs)(*centroid)

# tile_stack = stack.loc[..., y_utm+buffer:y_utm-buffer, x_utm-buffer:x_utm+buffer]
# tile_stack

# tile_stack_scaled = stack_scaled.loc[..., y_utm+buffer:y_utm-buffer, x_utm-buffer:x_utm+buffer]
# tile_stack

weekly_stack = weekly.loc[..., y_utm+buffer:y_utm-buffer, x_utm-buffer:x_utm+buffer]
weekly_stack

# weekly_stack_scaled = weekly_scaled.loc[..., y_utm+buffer:y_utm-buffer, x_utm-buffer:x_utm+buffer]
# weekly_stack_scaled


<xarray.DataArray 'stackstac-229cfab4ba93ad9a4b57c34300f511f5' (time: 22,
                                                                band: 5,
                                                                y: 2048, x: 2048)> Size: 4GB
dask.array<getitem, shape=(22, 5, 2048, 2048), dtype=float64, chunksize=(1, 1, 2048, 2048), chunktype=numpy.ndarray>
Coordinates: (12/20)
  * band                                     (band) <U3 60B 'B04' ... 'B11'
  * x                                        (x) float64 16kB 5.507e+05 ... 5...
  * y                                        (y) float64 16kB 7.635e+06 ... 7...
    s2:product_type                          <U7 28B 'S2MSI2A'
    s2:processing_baseline                   <U5 20B '02.12'
    s2:mgrs_tile                             <U5 20B '22WEB'
    ...                                       ...
    gsd                                      (band) float64 40B dask.array<chunksize=(5,), meta=np.ndarray>
    common_name                              (band) <U6 120B dask.array<chunksize=(5,), meta=np.ndarray>
    center_wavelength                        (band) float64 40B dask.array<chunksize=(5,), meta=np.ndarray>
    full_width_half_max                      (band) float64 40B dask.array<chunksize=(5,), meta=np.ndarray>
    epsg                                     int64 8B 32622
  * time                                     (time) datetime64[ns] 176B 2019-...
Attributes:
    spec:        RasterSpec(epsg=32622, bounds=(549970, 7613480, 571910, 7635...
    crs:         epsg:32622
    transform:   | 10.00, 0.00, 549970.00|\n| 0.00,-10.00, 7635420.00|\n| 0.0...
    resolution:  10

In [13]:
## PULL IN DESIRED STACK

stack_to_pull = weekly_stack

with dask.diagnostics.ProgressBar():
    stack_in_mem = stack_to_pull.compute()
print(f"shape of stack in memory: {stack_in_mem.shape}")

[########################################] | 100% Completed | 119.30 s
shape of stack in memory: (22, 5, 2048, 2048)


In [14]:
## SAVE THE TILES TO DISK

stack_name = "tiles_wk"

stack_in_mem = stack_in_mem.rename(f"{stack_name}")

# make a clean copy
tiles_clean = stack_in_mem.copy()

# 2a) drop any attr whose value isn’t a basic dtype
for k, v in list(tiles_clean.attrs.items()):
    if not isinstance(v, (str, int, float, list, tuple, np.ndarray)):
        tiles_clean.attrs.pop(k)

# 2b) clear out any leftover encoding metadata
tiles_clean.encoding.clear()

# Turn your DA into a Dataset
ds = tiles_clean.to_dataset()

# # See what variables you have and their dtypes:
# for vn in ds.variables:
#     print(vn, ds[vn].dtype)
# # You will almost certainly see one named `None` with dtype `object`

# Drop *any* object‐dtype variable
for vn in list(ds.variables):
    if ds[vn].dtype == object:
        ds = ds.drop_vars(vn)

# Now write
ds.to_zarr(f"/home/jupyter/{stack_name}.zarr", mode="w")


In [15]:
## TO OPEN .zarr FROM DISK

stack_name = "tiles_wk"
ds = xr.open_zarr(f"/home/jupyter/{stack_name}.zarr", chunks="auto")
tiles_reload = ds[f'{stack_name}']
tiles_reload

<xarray.DataArray 'tiles_wk' (time: 22, band: 5, y: 2048, x: 2048)> Size: 4GB
dask.array<open_dataset-tiles_wk, shape=(22, 5, 2048, 2048), dtype=float64, chunksize=(3, 1, 256, 512), chunktype=numpy.ndarray>
Coordinates: (12/19)
  * band                                     (band) <U3 60B 'B04' ... 'B11'
    center_wavelength                        (band) float64 40B dask.array<chunksize=(5,), meta=np.ndarray>
    common_name                              (band) <U6 120B dask.array<chunksize=(5,), meta=np.ndarray>
    constellation                            <U10 40B ...
    epsg                                     int64 8B ...
    full_width_half_max                      (band) float64 40B dask.array<chunksize=(5,), meta=np.ndarray>
    ...                                       ...
    s2:product_type                          <U7 28B ...
    s2:saturated_defective_pixel_percentage  float64 8B ...
  * time                                     (time) datetime64[ns] 176B 2019-...
    title                                    (band) <U26 520B dask.array<chunksize=(5,), meta=np.ndarray>
  * x                                        (x) float64 16kB 5.507e+05 ... 5...
  * y                                        (y) float64 16kB 7.635e+06 ... 7...
Attributes:
    crs:         epsg:32622
    resolution:  10
    transform:   [10.0, 0.0, 549970.0, 0.0, -10.0, 7635420.0, 0.0, 0.0, 1.0]